In [2]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
  Using cached panda-0.3.1.tar.gz (5.8 kB)
  Preparing metadata (setup.py) ... done
  Using cached rasterio-1.4.4-cp310-cp310-manylinux_2_28_x86_64.whl (35.3 MB)
  Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
  Using cached shapely-2.1.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.1 MB)
  Using cached affine-2.4.0-py3-none-any.whl (15 kB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached pyogrio-0.12.1-cp310-cp310-manylinux_2_28_x86_64.whl (32.4 MB)
  Using cached pyproj-3.7.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (9.3 MB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (2

In [3]:
import os
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import box
from rasterio.windows import from_bounds

def extract_features(csv_path, tif_path, out_csv, tag):
    df = pd.read_csv(csv_path)
    rows = []

    with rasterio.open(tif_path) as src:
        for _, cell in df.iterrows():
            bbox_4326 = box(
                cell["min_lon"],
                cell["min_lat"],
                cell["max_lon"],
                cell["max_lat"]
            )

            bbox = gpd.GeoSeries([bbox_4326], crs="EPSG:4326").to_crs(src.crs).iloc[0]
            minx, miny, maxx, maxy = bbox.bounds

            win = from_bounds(minx, miny, maxx, maxy, transform=src.transform)

            # Bands 1-9 = RGB + NDVI + SWIR
            data = src.read(list(range(1, 10)), window=win).astype(np.float32)
            flat = data.reshape(9, -1)

            valid = ~(np.all(flat == 0, axis=0))
            if valid.any():
                flat = flat[:, valid]

            feat = {
                "cell_id": int(cell["cell_id"]),
                "row": int(cell["row"]),
                "col": int(cell["col"]),
            }

            for i in range(9):
                band_index = i + 1
                feat[f"band{band_index}_mean_{tag}"] = float(np.mean(flat[i]))
                feat[f"band{band_index}_std_{tag}"] = float(np.std(flat[i]))

            rows.append(feat)

    out = pd.DataFrame(rows)
    out.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv} | shape={out.shape}")
    return out

In [5]:
extract_features(
    "run_with_noise_20260303_2355/Composition_diff_in_one_year/grid_stats_with_noise.csv",
    "run_with_noise_20260303_2355/Composition_diff_in_one_year/training_data_with_noise.tif",
    "features_noise_diff.csv",
    "diff"
)

Saved: features_noise_diff.csv | shape=(1600, 21)


,cell_id,row,col,band1_mean_diff,band1_std_diff,band2_mean_diff,band2_std_diff,band3_mean_diff,band3_std_diff,band4_mean_diff,...,band5_mean_diff,band5_std_diff,band6_mean_diff,band6_std_diff,band7_mean_diff,band7_std_diff,band8_mean_diff,band8_std_diff,band9_mean_diff,band9_std_diff
0,0,0,0,94.441299,47.565605,80.120598,27.147528,51.921200,22.276628,158.164505,...,196.403595,37.483620,0.0,0.0,170.220200,49.978981,243.256393,23.689459,94.948898,47.558830
1,1,0,1,101.252899,46.156986,81.111298,24.304808,53.610001,20.716566,181.781693,...,196.379898,33.384911,0.0,0.0,174.875595,52.637074,231.024399,31.015144,101.747704,46.134609
2,2,0,2,100.649498,43.026951,77.609901,25.206306,51.405701,20.964596,215.045395,...,205.827103,47.434929,0.0,0.0,161.046997,56.650158,208.658203,53.021774,101.133904,43.030624
3,3,0,3,54.661900,38.370583,51.835400,23.361496,30.210199,16.861359,92.312599,...,170.397400,43.535290,0.0,0.0,117.697701,65.888611,225.487396,46.258324,55.150002,38.378223
4,4,0,4,63.209400,28.062754,62.149502,17.174515,37.282101,17.287514,112.902000,...,178.793396,37.394905,0.0,0.0,138.863602,41.320091,230.598007,39.705528,63.704498,28.061289
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,1595,39,35,22.780714,4.533776,34.189106,4.612885,23.669464,2.881504,16.867144,...,136.400177,7.801994,0.0,0.0,45.832500,10.437799,194.641968,22.746031,23.302500,4.540602
1596,1596,39,36,23.568214,10.325869,34.732677,7.188617,23.863750,5.616637,21.039106,...,138.454468,16.467741,0.0,0.0,50.231785,19.302254,197.583221,24.250801,24.092857,10.319138
1597,1597,39,37,25.521965,11.906866,39.580002,12.083521,25.144464,6.037652,10.186071,...,133.066971,13.073513,0.0,0.0,62.140537,24.615276,232.255005,24.624580,26.045713,11.898756
1598,1598,39,38,28.205179,21.903788,37.966427,15.099075,25.203751,10.278119,27.397322,...,141.421600,27.778536,0.0,0.0,62.136250,42.983913,205.342850,26.112728,28.758751,21.891212


In [6]:
extract_features(
    "run_with_noise_20260303_2355/Composition_prediction_in_3_years/2020/grid_stats_with_noise.csv",
    "run_with_noise_20260303_2355/Composition_prediction_in_3_years/2020/training_data_with_noise.tif",
    "features_noise_2020.csv",
    "2020"
)

Saved: features_noise_2020.csv | shape=(1600, 21)


,cell_id,row,col,band1_mean_2020,band1_std_2020,band2_mean_2020,band2_std_2020,band3_mean_2020,band3_std_2020,band4_mean_2020,...,band5_mean_2020,band5_std_2020,band6_mean_2020,band6_std_2020,band7_mean_2020,band7_std_2020,band8_mean_2020,band8_std_2020,band9_mean_2020,band9_std_2020
0,0,0,0,206.748596,54.325974,207.444702,52.221867,196.628006,60.284481,235.663193,...,204.367706,35.423508,0.0,0.0,222.533493,39.619106,253.567993,7.154885,207.071899,54.168457
1,1,0,1,206.933701,57.237186,204.888901,57.748791,193.718903,64.707237,242.269806,...,197.931702,36.179352,0.0,0.0,224.034103,42.178242,253.380203,8.152340,207.253799,57.057487
2,2,0,2,128.336502,63.341843,128.919403,63.391739,113.314499,69.000778,205.833694,...,211.273193,35.155437,0.0,0.0,158.207901,55.083530,230.613007,36.463028,128.818207,63.281448
3,3,0,3,224.330994,45.235538,227.813293,42.746220,221.524002,50.915558,244.697601,...,206.216202,33.314011,0.0,0.0,224.373093,36.732735,254.828705,2.815272,224.568497,45.054958
4,4,0,4,95.613098,57.399864,108.212601,53.133549,88.479401,59.199299,128.728104,...,184.471497,40.613365,0.0,0.0,138.179199,50.047428,244.836395,24.132004,96.127403,57.364105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,1595,39,35,18.966608,7.482346,28.696251,6.893402,11.517143,4.636508,5.870179,...,130.921600,10.030477,0.0,0.0,40.473213,13.544978,193.641068,25.997644,19.463392,7.490166
1596,1596,39,36,18.184999,8.445290,27.305178,7.654450,10.565179,5.456029,7.568393,...,131.771072,12.245979,0.0,0.0,39.982143,12.979972,194.405182,26.719227,18.691965,8.449172
1597,1597,39,37,19.739107,9.613326,32.632500,13.252033,11.286428,6.482870,2.190179,...,129.087860,5.397433,0.0,0.0,52.016430,24.321333,227.776245,30.083124,20.219999,9.595841
1598,1598,39,38,21.951429,17.715574,31.251429,14.752724,11.548036,10.388826,14.168928,...,135.042496,20.322710,0.0,0.0,51.294285,32.818542,206.058578,32.042152,22.435892,17.709869


In [7]:
extract_features(
    "run_with_noise_20260303_2355/Composition_prediction_in_3_years/2021/grid_stats_with_noise.csv",
    "run_with_noise_20260303_2355/Composition_prediction_in_3_years/2021/training_data_with_noise.tif",
    "features_noise_2021.csv",
    "2021"
)

Saved: features_noise_2021.csv | shape=(1600, 21)


,cell_id,row,col,band1_mean_2021,band1_std_2021,band2_mean_2021,band2_std_2021,band3_mean_2021,band3_std_2021,band4_mean_2021,...,band5_mean_2021,band5_std_2021,band6_mean_2021,band6_std_2021,band7_mean_2021,band7_std_2021,band8_mean_2021,band8_std_2021,band9_mean_2021,band9_std_2021
0,0,0,0,96.435600,43.877716,82.218102,26.014826,53.563099,22.109129,165.071899,...,187.093994,34.268337,0.0,0.0,175.090393,56.739208,237.101807,25.269823,96.922203,43.860466
1,1,0,1,96.675301,38.564442,82.828598,20.418991,53.394299,17.794428,172.767700,...,189.819595,31.338409,0.0,0.0,177.554092,50.626595,231.178497,30.470802,97.167801,38.570450
2,2,0,2,84.064598,39.971001,75.831398,23.739422,47.804798,20.128658,159.795303,...,187.245407,42.876095,0.0,0.0,153.570801,61.562042,225.758408,52.982521,84.572304,39.950115
3,3,0,3,48.782902,26.071531,50.922199,17.130556,29.281200,11.883566,82.657997,...,165.589401,36.277737,0.0,0.0,112.184601,60.944901,228.409607,42.462559,49.301201,26.082716
4,4,0,4,66.009201,24.662022,64.049500,15.132398,39.092602,14.306153,122.919197,...,183.673401,34.827423,0.0,0.0,144.442200,42.766956,233.341003,37.777885,66.514900,24.663261
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,1595,39,35,25.682678,5.804357,35.421963,5.422241,20.331249,3.310172,20.426250,...,138.160721,9.380996,0.0,0.0,49.938572,12.490875,207.120712,21.111788,26.182501,5.825061
1596,1596,39,36,25.239820,11.611408,34.308037,7.832939,18.780714,6.257548,22.367500,...,139.000351,16.786230,0.0,0.0,52.764999,21.044609,207.166428,21.260202,25.720535,11.598690
1597,1597,39,37,30.965536,17.687567,39.331249,14.281047,21.928392,8.839141,20.808929,...,138.368927,18.153275,0.0,0.0,73.796074,39.992085,235.108749,23.526093,31.452679,17.668331
1598,1598,39,38,30.286964,22.294744,37.308392,14.927015,21.471071,10.879382,31.137678,...,143.298569,27.057390,0.0,0.0,65.626610,43.566360,211.497864,26.543327,30.795893,22.296555


In [8]:
import pandas as pd
import re

BAND_NAME = {
    1: "R",
    2: "G",
    3: "B",
    4: "NDVI_1",
    5: "NDVI_2",
    6: "NDVI_3",
    7: "SWIR_1",
    8: "SWIR_2",
    9: "SWIR_3",
}

def rename_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    new_cols = {}
    pattern = re.compile(r"^band(\d+)_(mean|std)_(.+)$")

    for c in df.columns:
        m = pattern.match(c)
        if not m:
            continue

        band_num = int(m.group(1))
        stat = m.group(2)
        tag = m.group(3)

        if band_num in BAND_NAME:
            new_cols[c] = f"{BAND_NAME[band_num]}_{stat}_{tag}"

    return df.rename(columns=new_cols)

In [9]:
fdiff = pd.read_csv("features_noise_diff.csv")
f20 = pd.read_csv("features_noise_2020.csv")
f21 = pd.read_csv("features_noise_2021.csv")

fdiff = rename_feature_columns(fdiff)
f20 = rename_feature_columns(f20)
f21 = rename_feature_columns(f21)

fdiff.to_csv("features_noise_diff_renamed.csv", index=False)
f20.to_csv("features_noise_2020_renamed.csv", index=False)
f21.to_csv("features_noise_2021_renamed.csv", index=False)

print("Saved all renamed files.")

Saved all renamed files.
